
# Profile the WDM transform architecture and the \(m=0\) convention

This notebook profiles the two waveform WDM transforms used before the prepared
one-arm response. It does **not** change any production source files.

It addresses seven questions:

1. What data representations, dtypes, layouts, and persistent arrays are used?
2. How much time belongs to one transform, two sequential transforms, and
   host/device conversion?
3. What does `calc_m0=True` actually add to the coefficient representation?
4. Does disabling the \(m=0\) calculation change the nonzero-frequency
   coefficients, runtime, memory, or one-arm response?
5. Does the WDM constructor parameter `q` affect only the \(m=0\) layer, or
   does it alter the \(m\geq1\) coefficients as well?
6. Can the plus/cross transforms be batched or vectorised through the current
   public `wdm.dwt` method?
7. Does the transform continue to scale across the available physical CPU
   cores, or does it plateau?

The CPU-scaling section is an indirect diagnostic. Without hardware counters it
cannot prove that memory bandwidth is the limiting resource, but it can
distinguish strong parallel scaling from an early shared-resource or
implementation-structure plateau.

Run this notebook in a fresh kernel from the `WDM_LISAresponse` repository
environment. The full-year waveform is generated only once in the main kernel,
then reused by all tests.


In [ ]:

# --- 1. Imports, CPU backend, and general helpers ---

import gc
import inspect
import json
import os
import platform
import shutil
import subprocess
import sys
import textwrap
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from pathlib import Path

# This must be set before importing JAX or packages that import JAX.
os.environ["JAX_PLATFORMS"] = "cpu"

import jax
import jax.numpy as jnp
import numpy as np

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import psutil
except ImportError:
    psutil = None

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

from few.utils.constants import YRSID_SI
from few.waveform import GenerateEMRIWaveform
from lisaorbits.file import read_orbit_file
from lisaorbits import InterpolatedOrbits

from WDM_LISAresponse.code.lisa_response.waveforms import (
    generate_windowed_padded_waveform,
)
from WDM_LISAresponse.code.lisa_response.response_with_plan import (
    _build_wdm_dT_grid,
    _build_wdm_dT_context_from_wdm,
)
from WDM_LISAresponse.code.lisa_response.one_arm_plan import (
    OneArmWDMPlan,
)

import WDM
import WDM_LISAresponse


PROCESS = psutil.Process(os.getpid()) if psutil is not None else None


def rss_gib():
    if PROCESS is None:
        return np.nan
    return PROCESS.memory_info().rss / 1024**3


def block_result(value):
    for leaf in jax.tree_util.tree_leaves(value):
        if hasattr(leaf, "block_until_ready"):
            leaf.block_until_ready()
    return value


def summarize_timings(values):
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        raise ValueError("At least one timing is required.")
    q25, q75 = np.percentile(values, [25.0, 75.0])
    return {
        "median_s": float(np.median(values)),
        "mean_s": float(np.mean(values)),
        "minimum_s": float(np.min(values)),
        "maximum_s": float(np.max(values)),
        "q25_s": float(q25),
        "q75_s": float(q75),
        "iqr_s": float(q75 - q25),
        "times_s": [float(x) for x in values],
    }


def benchmark_call(fn, *, warmups=2, repeats=15):
    result = None
    for _ in range(max(0, int(warmups))):
        result = fn()
        block_result(result)

    timings = []
    for _ in range(max(1, int(repeats))):
        started = time.perf_counter()
        result = fn()
        block_result(result)
        timings.append(time.perf_counter() - started)

    return result, summarize_timings(timings)


def compare_arrays(candidate, reference):
    candidate = jnp.asarray(candidate)
    reference = jnp.asarray(reference)
    if candidate.shape != reference.shape:
        raise ValueError(
            f"Shape mismatch: candidate={candidate.shape}, reference={reference.shape}"
        )

    delta = candidate - reference
    max_abs = jnp.max(jnp.abs(delta))
    reference_norm = jnp.linalg.norm(reference)
    rel_l2 = jnp.where(
        reference_norm > 0,
        jnp.linalg.norm(delta) / reference_norm,
        jnp.nan,
    )
    max_abs.block_until_ready()
    rel_l2.block_until_ready()
    return {
        "max_abs": float(max_abs),
        "rel_l2": float(rel_l2),
    }


def show_rows(rows, *, sort_by=None):
    rows = list(rows)
    if pd is not None:
        frame = pd.DataFrame(rows)
        if sort_by is not None and sort_by in frame.columns:
            frame = frame.sort_values(sort_by, kind="stable").reset_index(drop=True)
        display(frame)
        return frame

    if sort_by is not None:
        rows = sorted(rows, key=lambda row: row.get(sort_by, np.inf))
    for row in rows:
        print(row)
    return rows


print("Timestamp:", datetime.now().isoformat(timespec="seconds"))
print("Python executable:", sys.executable)
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("JAX:", jax.__version__)
print("JAX backend:", jax.default_backend())
print("JAX devices:", jax.devices())
print("WDM:", WDM.__file__)
print("WDM_LISAresponse:", WDM_LISAresponse.__file__)
print("Current affinity:", sorted(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else "unavailable")
print("RSS:", f"{rss_gib():.2f} GiB")

assert jax.default_backend() == "cpu"
assert all(device.platform == "cpu" for device in jax.devices())


In [ ]:

# --- 2. Scientific configuration and benchmark controls ---

# Full-year production-like case.
N = 3_200_000
dt = 10
Nf = 100
q_wdm = 4
d_wdm = 4
A_frac = 0.25

# Windowing.
alpha = 0.005
tukey_factor = 4

# Orbit start.
ORBIT_FILE = "esa-trailing-orbits.h5"

# EMRI parameters.
theta = np.pi / 2
phi = 0.0
m1 = 1e6
m2 = 10.0
a = 0.9
p0 = 8.5
e0 = 0.3
x0 = 1.0
dist = 1.0
qK = np.pi / 3
phiK = np.pi / 3
Phi_phi0 = np.pi / 3
Phi_theta0 = 0.0
Phi_r0 = np.pi / 3
eps = 1e-2

# Timings.
WARMUPS = 2
REPEATS = 15
MIN_INTERESTING_IMPROVEMENT_PERCENT = 3.0
MAX_REL_L2 = 1e-6


# m=0 / q convention study.
RUN_M0_Q_STUDY = True
M0_WARMUPS = 2
M0_REPEATS = 15
M0_Q_VALUES = sorted({1, 2, 4, 8, q_wdm})
M0_Q_WARMUPS = 1
M0_Q_REPEATS = 5

# Optional one-arm response comparison. This uses the current production
# shift settings and reports the response change caused by excluding m=0.
RUN_M0_RESPONSE_CHECK = True
M0_RESPONSE_WARMUPS = 2
M0_RESPONSE_REPEATS = 9
M0_L_TRUNC = 25
M0_SHIFT_KWARGS = {
    "assembly_backend": "lagfirst_chunked_lagblock",
    "assembly_precision": "complex64",
    "tl_tp_mode": "interp",
    "tl_tp_interp_points": 32,
    "tl_tp_interp_kind": "linear",
    "use_jax": True,
    "row_chunk_size": 32640,
    "lag_block_size": 17,
    "job_block_size": 1,
}

# Candidate diagnostics.
RUN_NATIVE_BATCH_TEST = True
RUN_VMAP_TEST = True
RUN_THREAD_TEST = True
RUN_BATCH_SCALING = False

# Fresh-process CPU scaling. This does not require Linux perf.
RUN_CPU_SCALING = True
CPU_SCALING_WARMUPS = 2
CPU_SCALING_REPEATS = 9
REQUESTED_CORE_COUNTS = None  # None -> 1,2,4,8,...,all available physical cores.

CACHE_DIR = Path(".benchmark_cache") / "wdm_transform_profile"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Full-year samples:", N)
print("Expected padded WDM Nt:", N // (2 * Nf))
print("Expected input bytes per polarization:", f"{N * 8 / 1024**2:.1f} MiB for float64")
print("Main benchmark warmups/repeats:", WARMUPS, REPEATS)
print("m=0 / q study enabled:", RUN_M0_Q_STUDY)
print("m=0 one-arm response check enabled:", RUN_M0_RESPONSE_CHECK)
print("q values:", M0_Q_VALUES)
print("CPU scaling enabled:", RUN_CPU_SCALING)



## 3. Generate the fixed full-year waveform

Waveform generation and tapering are not included in the WDM-transform timings.
The resulting padded plus/cross arrays are reused throughout the notebook.


In [ ]:

# --- 3. Generate and pad the waveform ---

td_gen = GenerateEMRIWaveform(
    "FastKerrEccentricEquatorialFlux",
    sum_kwargs=dict(pad_output=True, odd_len=True),
    return_list=True,
)

orbit_data = read_orbit_file(ORBIT_FILE)
orbit_times = orbit_data.t0 + orbit_data.dt * np.arange(orbit_data.size)
orbits_trailing = InterpolatedOrbits(
    orbit_times,
    orbit_data.positions,
    orbit_data.velocities,
)
t0 = orbit_data.t0 + 1.0 * YRSID_SI

model_kwargs = {
    "td_gen": td_gen,
    "m1": m1,
    "m2": m2,
    "a": a,
    "p0": p0,
    "e0": e0,
    "x0": x0,
    "dist": dist,
    "theta": theta,
    "phi": phi,
    "qK": qK,
    "phiK": phiK,
    "Phi_phi0": Phi_phi0,
    "Phi_theta0": Phi_theta0,
    "Phi_r0": Phi_r0,
    "eps": eps,
}

window_kwargs = {
    "alpha": alpha,
    "tukey_factor": tukey_factor,
}

started = time.perf_counter()
times_pad, hplus_td, hcross_td, pad_info = generate_windowed_padded_waveform(
    waveform_model="emri",
    t0=t0,
    dt=dt,
    N=N,
    model_kwargs=model_kwargs,
    window_function="tukey",
    window_kwargs=window_kwargs,
    wdm_Nf=Nf,
    return_padding_info=True,
)
waveform_generation_s = time.perf_counter() - started

print("Orbit samples:", orbit_data.size)
print("Waveform generation_s:", f"{waveform_generation_s:.3f}")
print("times_pad:", times_pad.shape, times_pad.dtype)
print("hplus_td:", hplus_td.shape, hplus_td.dtype)
print("hcross_td:", hcross_td.shape, hcross_td.dtype)
print("WDM Nt x Nf:", pad_info["wdm_Nt"], "x", Nf)
print("Tail right padding:", pad_info["wdm_tail_pad_right"])
print("RSS:", f"{rss_gib():.2f} GiB")



## 4. Build the WDM object without transforming the waveform

Grid construction is measured separately because it is normally reusable across
many waveform evaluations.


In [ ]:

# --- 4. Build the reusable WDM grid ---

started = time.perf_counter()
grid_context = _build_wdm_dT_grid(
    times_pad,
    Nf=Nf,
    d=d_wdm,
    A_frac=A_frac,
    q=q_wdm,
    calc_m0=True,
)
grid_build_s = time.perf_counter() - started
wdm = grid_context["wdm"]

print("WDM grid build_s:", f"{grid_build_s:.3f}")
print("WDM type:", type(wdm))
print("WDM N:", wdm.N)
print("WDM Nt:", wdm.Nt)
print("WDM Nf:", wdm.Nf)
print("WDM dt:", wdm.dt)
print("WDM dT:", wdm.dT)
print("WDM dF:", wdm.dF)
print("WDM q:", getattr(wdm, "q", None))
print("WDM d:", getattr(wdm, "d", None))
print("WDM calc_m0:", getattr(wdm, "calc_m0", None))
print("RSS:", f"{rss_gib():.2f} GiB")



## 5. Inventory the data representation

This section reports array type, dtype, shape, byte size, strides, contiguity,
and device placement. It also inventories array-like state stored on the WDM
object.

A large resident array does not by itself prove memory-bandwidth limitation.
The inventory identifies which representation changes are even plausible.


In [ ]:

# --- 5. Array and WDM-object representation inventory ---

def array_record(name, value):
    record = {"name": name, "python_type": type(value).__name__}

    try:
        shape = tuple(value.shape)
        dtype = str(value.dtype)
    except Exception:
        record.update({
            "shape": None,
            "dtype": None,
            "nbytes_mib": np.nan,
            "location": "non-array",
            "c_contiguous": None,
            "f_contiguous": None,
            "strides": None,
        })
        return record

    record["shape"] = shape
    record["dtype"] = dtype

    try:
        record["nbytes_mib"] = float(value.nbytes / 1024**2)
    except Exception:
        record["nbytes_mib"] = float(np.prod(shape) * np.dtype(value.dtype).itemsize / 1024**2)

    is_jax = hasattr(value, "devices") or type(value).__module__.startswith(("jax", "jaxlib"))
    record["location"] = "jax-device" if is_jax else "host"

    if isinstance(value, np.ndarray):
        record["c_contiguous"] = bool(value.flags.c_contiguous)
        record["f_contiguous"] = bool(value.flags.f_contiguous)
        record["strides"] = tuple(int(x) for x in value.strides)
    else:
        record["c_contiguous"] = None
        record["f_contiguous"] = None
        record["strides"] = None

    return record


representation_rows = [
    array_record("times_pad", times_pad),
    array_record("hplus_td", hplus_td),
    array_record("hcross_td", hcross_td),
]

known_wdm_arrays = [
    "window_FD",
    "window_TD",
    "freqs",
    "times",
    "phi",
    "phif",
    "w0",
    "w1",
]

seen_ids = set()
for attr_name in known_wdm_arrays:
    if hasattr(wdm, attr_name):
        value = getattr(wdm, attr_name)
        if hasattr(value, "shape") and id(value) not in seen_ids:
            seen_ids.add(id(value))
            representation_rows.append(array_record(f"wdm.{attr_name}", value))

for attr_name in sorted(dir(wdm)):
    if attr_name.startswith("_") or attr_name in known_wdm_arrays:
        continue
    try:
        value = getattr(wdm, attr_name)
    except Exception:
        continue
    if callable(value) or not hasattr(value, "shape") or id(value) in seen_ids:
        continue
    seen_ids.add(id(value))
    representation_rows.append(array_record(f"wdm.{attr_name}", value))

representation_table = show_rows(representation_rows, sort_by="nbytes_mib")

persistent_wdm_mib = float(np.nansum([
    row["nbytes_mib"]
    for row in representation_rows
    if row["name"].startswith("wdm.")
]))

expected_output_mib = (
    int(wdm.Nt) * int(wdm.Nf) * np.dtype(np.complex128).itemsize / 1024**2
)

print("Persistent array-like WDM state:", f"{persistent_wdm_mib:.1f} MiB")
print("One complex128 WDM output:", f"{expected_output_mib:.1f} MiB")
print("Two complex128 WDM outputs:", f"{2 * expected_output_mib:.1f} MiB")
print("RSS:", f"{rss_gib():.2f} GiB")



## 6. Inspect the current `dwt` implementation

The source inspection is intentionally descriptive. It looks for FFT calls,
Python loops, JAX transformations, NumPy conversions, and explicit allocation.
These signals guide later source-level work but do not replace timing.


In [ ]:

# --- 6. Public DWT source and structural heuristics ---

dwt_method = getattr(type(wdm), "dwt", None)
print("Bound method:", wdm.dwt)
print("Signature:", inspect.signature(wdm.dwt))
print("Defined on:", type(wdm))
print("Source file:", inspect.getsourcefile(dwt_method) if dwt_method is not None else None)
print("Has .lower:", hasattr(wdm.dwt, "lower"))
print("Has .trace:", hasattr(wdm.dwt, "trace"))

try:
    dwt_source = inspect.getsource(dwt_method)
except Exception as exc:
    dwt_source = ""
    print("Could not read DWT source:", repr(exc))

if dwt_source:
    source_lines = dwt_source.splitlines()
    print("\n--- dwt source (first 180 lines) ---")
    print("\n".join(source_lines[:180]))
    if len(source_lines) > 180:
        print(f"... {len(source_lines) - 180} additional lines omitted")

    lowered = dwt_source.lower()
    source_signals = {
        "fft_or_rfft": any(token in lowered for token in ("fft(", "rfft(", "fft.", "rfft.")),
        "jax_jit": "jax.jit" in lowered or "@jit" in lowered or "@jax.jit" in lowered,
        "jax_vmap": "vmap" in lowered,
        "jax_scan_or_loop": any(token in lowered for token in ("lax.scan", "fori_loop", "while_loop")),
        "python_for": "for " in lowered,
        "numpy_conversion": "np.asarray" in lowered or "numpy.asarray" in lowered,
        "jax_conversion": "jnp.asarray" in lowered,
        "explicit_zeros": "zeros(" in lowered or "empty(" in lowered,
        "concatenate_or_stack": "concatenate(" in lowered or "stack(" in lowered,
    }
    print("\n--- structural signals ---")
    for key, value in source_signals.items():
        print(f"{key}: {value}")
else:
    source_signals = {}

if dwt_source:
    import re
    private_call_names = sorted(set(
        token
        for token in re.findall(r"self\.(_[A-Za-z0-9_]+)\s*\(", dwt_source)
    ))
    print("\nPrivate methods called directly:", private_call_names)

    for method_name in private_call_names[:8]:
        method = getattr(type(wdm), method_name, None)
        if method is None:
            continue
        try:
            method_source = inspect.getsource(method)
        except Exception:
            continue
        print(f"\n--- {method_name} source (first 120 lines) ---")
        print("\n".join(method_source.splitlines()[:120]))



## 7. Baseline transform timings

The plus and cross transforms are measured separately and together. All calls
are warmed and synchronized before timing.

The two-transform timing is the production-relevant baseline.


In [ ]:

# --- 7. Baseline one- and two-transform timings ---

def run_plus_host():
    return wdm.dwt(hplus_td)


def run_cross_host():
    return wdm.dwt(hcross_td)


def run_two_sequential_host():
    return wdm.dwt(hplus_td), wdm.dwt(hcross_td)


rss_before_dwt = rss_gib()

w_plus_ref, plus_timing = benchmark_call(
    run_plus_host,
    warmups=WARMUPS,
    repeats=REPEATS,
)
w_cross_ref, cross_timing = benchmark_call(
    run_cross_host,
    warmups=WARMUPS,
    repeats=REPEATS,
)
two_ref, two_timing = benchmark_call(
    run_two_sequential_host,
    warmups=WARMUPS,
    repeats=REPEATS,
)

rss_after_dwt = rss_gib()

block_result((w_plus_ref, w_cross_ref))
reference_stack = jnp.stack(
    (jnp.asarray(w_plus_ref), jnp.asarray(w_cross_ref)),
    axis=0,
)
reference_stack.block_until_ready()

baseline_rows = [
    {"route": "plus_only", **plus_timing},
    {"route": "cross_only", **cross_timing},
    {"route": "two_sequential", **two_timing},
]
baseline_table = show_rows(baseline_rows)

print("\nOutput representations:")
show_rows([
    array_record("w_plus_ref", w_plus_ref),
    array_record("w_cross_ref", w_cross_ref),
    array_record("reference_stack", reference_stack),
])

single_sum = plus_timing["median_s"] + cross_timing["median_s"]
print("\nSum of separate medians:", f"{single_sum:.4f} s")
print("Two-transform median:", f"{two_timing['median_s']:.4f} s")
print(
    "Two-transform difference from summed medians:",
    f"{100.0 * (two_timing['median_s'] / single_sum - 1.0):+.2f}%",
)
print("RSS before DWT:", f"{rss_before_dwt:.2f} GiB")
print("RSS after DWT:", f"{rss_after_dwt:.2f} GiB")
print("RSS change:", f"{rss_after_dwt - rss_before_dwt:+.2f} GiB")



## 8. Test the \(m=0\) convention and the constructor parameter `q`

The production-like transform above uses:

```python
calc_m0=True
```

This section builds a second transform with `calc_m0=False` and asks:

- whether the output shape changes;
- whether the first coefficient column is retained, zeroed, or omitted;
- how much coefficient power lies in the \(m=0\) layer;
- whether the \(m\geq1\) coefficients remain unchanged;
- how much time and persistent state are saved;
- whether varying constructor `q` changes only \(m=0\) or the entire basis.

The alignment helper deliberately tests multiple plausible output conventions.
It does not assume in advance that `calc_m0=False` preserves an empty first
column.


In [ ]:

# --- 8a. calc_m0=True versus calc_m0=False ---

def scalar_rel_l2(candidate, reference):
    candidate = jnp.asarray(candidate)
    reference = jnp.asarray(reference)

    if tuple(candidate.shape) != tuple(reference.shape):
        return np.nan

    delta = candidate - reference
    reference_norm = jnp.linalg.norm(reference)
    value = jnp.where(
        reference_norm > 0,
        jnp.linalg.norm(delta) / reference_norm,
        jnp.nan,
    )
    value.block_until_ready()
    return float(value)


def component_power_fraction(coefficients, column=0):
    coefficients = jnp.asarray(coefficients)
    if coefficients.ndim < 1 or coefficients.shape[-1] <= column:
        return np.nan

    total = jnp.sum(jnp.abs(coefficients) ** 2)
    selected = jnp.sum(jnp.abs(coefficients[..., column]) ** 2)
    value = jnp.where(total > 0, selected / total, jnp.nan)
    value.block_until_ready()
    return float(value)


def best_nonzero_frequency_alignment(reference_with_m0, candidate_without_m0):
    """Compare plausible calc_m0=False output conventions.

    Tested conventions
    ------------------
    preserved_column:
        Both outputs have the same width and column zero remains reserved.
        Compare reference[..., 1:] with candidate[..., 1:].

    packed_nonzero:
        Both outputs have the same width but the no-m0 transform packs
        nonzero-frequency coefficients from column zero.
        Compare reference[..., 1:] with candidate[..., :-1].

    omitted_column:
        The no-m0 output has one fewer column.
        Compare reference[..., 1:] with candidate.
    """

    reference = jnp.asarray(reference_with_m0)
    candidate = jnp.asarray(candidate_without_m0)

    rows = []
    reference_width = int(reference.shape[-1])
    candidate_width = int(candidate.shape[-1])

    if reference.shape[:-1] != candidate.shape[:-1]:
        return {
            "best_convention": "incompatible_nonfrequency_shape",
            "best_rel_l2": np.nan,
            "candidates": [],
        }

    if reference_width == candidate_width and reference_width >= 2:
        rows.append({
            "convention": "preserved_column",
            "rel_l2": scalar_rel_l2(
                candidate[..., 1:],
                reference[..., 1:],
            ),
            "reference_slice": "[..., 1:]",
            "candidate_slice": "[..., 1:]",
        })
        rows.append({
            "convention": "packed_nonzero",
            "rel_l2": scalar_rel_l2(
                candidate[..., :-1],
                reference[..., 1:],
            ),
            "reference_slice": "[..., 1:]",
            "candidate_slice": "[..., :-1]",
        })

    if candidate_width == reference_width - 1:
        rows.append({
            "convention": "omitted_column",
            "rel_l2": scalar_rel_l2(
                candidate,
                reference[..., 1:],
            ),
            "reference_slice": "[..., 1:]",
            "candidate_slice": "[...]",
        })

    finite_rows = [
        row for row in rows
        if np.isfinite(row["rel_l2"])
    ]

    if not finite_rows:
        return {
            "best_convention": "unresolved",
            "best_rel_l2": np.nan,
            "candidates": rows,
        }

    best = min(finite_rows, key=lambda row: row["rel_l2"])
    return {
        "best_convention": best["convention"],
        "best_rel_l2": float(best["rel_l2"]),
        "candidates": rows,
    }


m0_study_error = None
grid_context_no_m0 = None
wdm_no_m0 = None
w_plus_no_m0 = None
w_cross_no_m0 = None
m0_timing_rows = []
m0_alignment_rows = []
m0_comparison = {}
m0_true_pair_times = []
m0_false_pair_times = []

if RUN_M0_Q_STUDY:
    try:
        build_started = time.perf_counter()
        grid_context_no_m0 = _build_wdm_dT_grid(
            times_pad,
            Nf=Nf,
            d=d_wdm,
            A_frac=A_frac,
            q=q_wdm,
            calc_m0=False,
        )
        grid_no_m0_build_s = time.perf_counter() - build_started
        wdm_no_m0 = grid_context_no_m0["wdm"]

        def run_two_with_m0():
            return wdm.dwt(hplus_td), wdm.dwt(hcross_td)

        def run_two_without_m0():
            return (
                wdm_no_m0.dwt(hplus_td),
                wdm_no_m0.dwt(hcross_td),
            )

        # Compile/warm both paths before paired timing.
        for _ in range(M0_WARMUPS):
            block_result(run_two_with_m0())
            block_result(run_two_without_m0())

        for repeat_index in range(M0_REPEATS):
            order = (
                ("with_m0", run_two_with_m0),
                ("without_m0", run_two_without_m0),
            )
            if repeat_index % 2:
                order = tuple(reversed(order))

            for label, function in order:
                started = time.perf_counter()
                output = function()
                block_result(output)
                elapsed = time.perf_counter() - started

                if label == "with_m0":
                    m0_true_pair_times.append(float(elapsed))
                else:
                    m0_false_pair_times.append(float(elapsed))

        w_plus_no_m0, w_cross_no_m0 = run_two_without_m0()
        block_result((w_plus_no_m0, w_cross_no_m0))

        with_m0_summary = summarize_timings(m0_true_pair_times)
        without_m0_summary = summarize_timings(m0_false_pair_times)

        plus_alignment = best_nonzero_frequency_alignment(
            w_plus_ref,
            w_plus_no_m0,
        )
        cross_alignment = best_nonzero_frequency_alignment(
            w_cross_ref,
            w_cross_no_m0,
        )

        m0_alignment_rows = [
            {
                "polarization": "plus",
                "with_m0_shape": tuple(jnp.asarray(w_plus_ref).shape),
                "without_m0_shape": tuple(jnp.asarray(w_plus_no_m0).shape),
                "best_convention": plus_alignment["best_convention"],
                "best_nonzero_rel_l2": plus_alignment["best_rel_l2"],
                "all_alignment_candidates": plus_alignment["candidates"],
            },
            {
                "polarization": "cross",
                "with_m0_shape": tuple(jnp.asarray(w_cross_ref).shape),
                "without_m0_shape": tuple(jnp.asarray(w_cross_no_m0).shape),
                "best_convention": cross_alignment["best_convention"],
                "best_nonzero_rel_l2": cross_alignment["best_rel_l2"],
                "all_alignment_candidates": cross_alignment["candidates"],
            },
        ]

        m0_timing_rows = [
            {
                "route": "calc_m0_true",
                "grid_build_s": grid_build_s,
                **with_m0_summary,
            },
            {
                "route": "calc_m0_false",
                "grid_build_s": grid_no_m0_build_s,
                **without_m0_summary,
            },
        ]

        m0_comparison = {
            "with_m0_plus_power_fraction": component_power_fraction(
                w_plus_ref,
                0,
            ),
            "with_m0_cross_power_fraction": component_power_fraction(
                w_cross_ref,
                0,
            ),
            "without_m0_plus_first_column_power_fraction": (
                component_power_fraction(w_plus_no_m0, 0)
            ),
            "without_m0_cross_first_column_power_fraction": (
                component_power_fraction(w_cross_no_m0, 0)
            ),
            "with_m0_output_shape": tuple(jnp.asarray(w_plus_ref).shape),
            "without_m0_output_shape": tuple(
                jnp.asarray(w_plus_no_m0).shape
            ),
            "with_m0_output_dtype": str(jnp.asarray(w_plus_ref).dtype),
            "without_m0_output_dtype": str(
                jnp.asarray(w_plus_no_m0).dtype
            ),
            "runtime_change_percent": 100.0 * (
                without_m0_summary["median_s"]
                / with_m0_summary["median_s"]
                - 1.0
            ),
            "speedup": (
                with_m0_summary["median_s"]
                / without_m0_summary["median_s"]
            ),
            "plus_nonzero_alignment": plus_alignment,
            "cross_nonzero_alignment": cross_alignment,
        }

        print("calc_m0 timing:")
        show_rows(m0_timing_rows)

        print("\nOutput alignment:")
        show_rows([
            {
                key: value
                for key, value in row.items()
                if key != "all_alignment_candidates"
            }
            for row in m0_alignment_rows
        ])

        print("\nDetailed alignment candidates:")
        for row in m0_alignment_rows:
            print(row["polarization"], row["all_alignment_candidates"])

        print("\nm=0 / first-column power fractions:")
        for key, value in m0_comparison.items():
            if "power_fraction" in key:
                print(f"  {key}: {value:.6e}")

        print(
            "\nTwo-DWT runtime change:",
            f"{m0_comparison['runtime_change_percent']:+.2f}%",
        )
        print("Speedup:", f"{m0_comparison['speedup']:.3f}x")
        print("calc_m0=False WDM type:", type(wdm_no_m0))
        print(
            "calc_m0=False WDM attributes:",
            {
                name: getattr(wdm_no_m0, name)
                for name in ("N", "Nt", "Nf", "q", "calc_m0", "dF", "dT")
                if hasattr(wdm_no_m0, name)
            },
        )
    except Exception as exc:
        m0_study_error = repr(exc)
        print("calc_m0 comparison failed:")
        print(m0_study_error)
else:
    print("RUN_M0_Q_STUDY=False; skipping.")



### 8b. Sweep constructor `q`

For each `q`, the notebook builds both `calc_m0=True` and `calc_m0=False`
transforms and compares their coefficients against the corresponding
`q=q_wdm` reference.

The most important columns are:

- `all_coeff_rel_l2`: change in the complete stored coefficient array;
- `best_nonzero_rel_l2`: smallest difference under the plausible nonzero-bin
  alignments;
- `first_column_power_fraction`: how much power sits in the first stored
  frequency column;
- `median_two_dwt_s`: transform cost.

If `calc_m0=False` gives `best_nonzero_rel_l2` near zero for all tested `q`,
then `q` is not materially affecting the nonzero-frequency representation.


In [ ]:

# --- 8b. q-dependence sweep ---

m0_q_rows = []

if RUN_M0_Q_STUDY and m0_study_error is None:
    reference_by_calc_m0 = {
        True: (w_plus_ref, w_cross_ref),
        False: (w_plus_no_m0, w_cross_no_m0),
    }

    for calc_m0_value in (True, False):
        for q_value in M0_Q_VALUES:
            print(
                f"\nq={q_value}, calc_m0={calc_m0_value}"
            )

            try:
                is_existing_reference = (
                    q_value == q_wdm
                    and (
                        (calc_m0_value and wdm is not None)
                        or (
                            not calc_m0_value
                            and wdm_no_m0 is not None
                        )
                    )
                )

                if is_existing_reference:
                    grid_candidate = (
                        grid_context
                        if calc_m0_value
                        else grid_context_no_m0
                    )
                    wdm_candidate = grid_candidate["wdm"]
                    build_s = (
                        grid_build_s
                        if calc_m0_value
                        else grid_no_m0_build_s
                    )
                else:
                    build_started = time.perf_counter()
                    grid_candidate = _build_wdm_dT_grid(
                        times_pad,
                        Nf=Nf,
                        d=d_wdm,
                        A_frac=A_frac,
                        q=int(q_value),
                        calc_m0=bool(calc_m0_value),
                    )
                    build_s = time.perf_counter() - build_started
                    wdm_candidate = grid_candidate["wdm"]

                def run_candidate_pair(
                    wdm_candidate=wdm_candidate,
                ):
                    return (
                        wdm_candidate.dwt(hplus_td),
                        wdm_candidate.dwt(hcross_td),
                    )

                candidate_pair, timing = benchmark_call(
                    run_candidate_pair,
                    warmups=M0_Q_WARMUPS,
                    repeats=M0_Q_REPEATS,
                )
                candidate_plus, candidate_cross = candidate_pair
                block_result(candidate_pair)

                reference_plus, reference_cross = (
                    reference_by_calc_m0[calc_m0_value]
                )

                plus_direct = scalar_rel_l2(
                    candidate_plus,
                    reference_plus,
                )
                cross_direct = scalar_rel_l2(
                    candidate_cross,
                    reference_cross,
                )

                plus_alignment = best_nonzero_frequency_alignment(
                    reference_plus,
                    candidate_plus,
                )
                cross_alignment = best_nonzero_frequency_alignment(
                    reference_cross,
                    candidate_cross,
                )

                m0_q_rows.append({
                    "q": int(q_value),
                    "calc_m0": bool(calc_m0_value),
                    "grid_build_s": float(build_s),
                    "median_two_dwt_s": timing["median_s"],
                    "iqr_two_dwt_s": timing["iqr_s"],
                    "plus_all_coeff_rel_l2": plus_direct,
                    "cross_all_coeff_rel_l2": cross_direct,
                    "plus_best_nonzero_rel_l2": (
                        plus_alignment["best_rel_l2"]
                    ),
                    "cross_best_nonzero_rel_l2": (
                        cross_alignment["best_rel_l2"]
                    ),
                    "plus_alignment_convention": (
                        plus_alignment["best_convention"]
                    ),
                    "cross_alignment_convention": (
                        cross_alignment["best_convention"]
                    ),
                    "plus_first_column_power_fraction": (
                        component_power_fraction(candidate_plus, 0)
                    ),
                    "cross_first_column_power_fraction": (
                        component_power_fraction(candidate_cross, 0)
                    ),
                    "output_shape": tuple(
                        jnp.asarray(candidate_plus).shape
                    ),
                    "output_dtype": str(
                        jnp.asarray(candidate_plus).dtype
                    ),
                    "error": None,
                })

                if not is_existing_reference:
                    del candidate_pair
                    del candidate_plus
                    del candidate_cross
                    del wdm_candidate
                    del grid_candidate
                    gc.collect()

            except Exception as exc:
                m0_q_rows.append({
                    "q": int(q_value),
                    "calc_m0": bool(calc_m0_value),
                    "grid_build_s": np.nan,
                    "median_two_dwt_s": np.nan,
                    "iqr_two_dwt_s": np.nan,
                    "plus_all_coeff_rel_l2": np.nan,
                    "cross_all_coeff_rel_l2": np.nan,
                    "plus_best_nonzero_rel_l2": np.nan,
                    "cross_best_nonzero_rel_l2": np.nan,
                    "plus_alignment_convention": None,
                    "cross_alignment_convention": None,
                    "plus_first_column_power_fraction": np.nan,
                    "cross_first_column_power_fraction": np.nan,
                    "output_shape": None,
                    "output_dtype": None,
                    "error": repr(exc),
                })
                print("  failed:", repr(exc))

m0_q_table = show_rows(
    m0_q_rows,
    sort_by="calc_m0",
)



## 9. Optional one-arm response comparison

Transform-level equality is not enough to decide whether \(m=0\) can be
removed. This section attempts to build the normal prepared one-arm response
for both WDM conventions and compares their six-link outputs.

The comparison reports:

- whether the no-\(m=0\) representation is accepted by the current response;
- one-arm plan-build times;
- warmed shift runtimes;
- relative \(L_2\) and maximum absolute response differences;
- estimated two-DWT-plus-response runtime.

This is not a replacement for the final time-domain or TDI mismatch validation.
It is an intermediate integration test.


In [ ]:

# --- 9. One-arm response integration check ---

m0_response_result = {
    "attempted": bool(RUN_M0_RESPONSE_CHECK),
    "supported": False,
    "error": None,
}
m0_response_pair_rows = []

if (
    RUN_M0_RESPONSE_CHECK
    and RUN_M0_Q_STUDY
    and m0_study_error is None
):
    try:
        def build_m0_response_plan(context):
            started = time.perf_counter()
            plan = OneArmWDMPlan.build_from_context(
                context,
                orbits_trailing,
                theta,
                phi,
                L_trunc=M0_L_TRUNC,
                variable_shift_kwargs=M0_SHIFT_KWARGS,
                use_jax=True,
            )
            return plan, time.perf_counter() - started

        response_plan_with_m0, response_plan_with_m0_build_s = (
            build_m0_response_plan(grid_context)
        )
        response_plan_without_m0, response_plan_without_m0_build_s = (
            build_m0_response_plan(grid_context_no_m0)
        )

        w_plus_with_m0_device = jnp.asarray(
            w_plus_ref,
            dtype=jnp.complex64,
        )
        w_cross_with_m0_device = jnp.asarray(
            w_cross_ref,
            dtype=jnp.complex64,
        )
        w_plus_without_m0_device = jnp.asarray(
            w_plus_no_m0,
            dtype=jnp.complex64,
        )
        w_cross_without_m0_device = jnp.asarray(
            w_cross_no_m0,
            dtype=jnp.complex64,
        )
        block_result((
            w_plus_with_m0_device,
            w_cross_with_m0_device,
            w_plus_without_m0_device,
            w_cross_without_m0_device,
        ))

        def run_response_with_m0():
            return (
                response_plan_with_m0
                .apply_batch_device_materialized(
                    w_plus_with_m0_device,
                    w_cross_with_m0_device,
                )
            )

        def run_response_without_m0():
            return (
                response_plan_without_m0
                .apply_batch_device_materialized(
                    w_plus_without_m0_device,
                    w_cross_without_m0_device,
                )
            )

        for _ in range(M0_RESPONSE_WARMUPS):
            block_result(run_response_with_m0())
            block_result(run_response_without_m0())

        with_m0_times = []
        without_m0_times = []

        for repeat_index in range(M0_RESPONSE_REPEATS):
            order = (
                ("with_m0", run_response_with_m0),
                ("without_m0", run_response_without_m0),
            )
            if repeat_index % 2:
                order = tuple(reversed(order))

            for label, function in order:
                started = time.perf_counter()
                output = function()
                block_result(output)
                elapsed = time.perf_counter() - started

                if label == "with_m0":
                    with_m0_times.append(float(elapsed))
                else:
                    without_m0_times.append(float(elapsed))

        response_with_m0 = run_response_with_m0()
        response_without_m0 = run_response_without_m0()
        block_result((response_with_m0, response_without_m0))

        response_comparison = compare_arrays(
            response_without_m0,
            response_with_m0,
        )
        with_m0_response_summary = summarize_timings(with_m0_times)
        without_m0_response_summary = summarize_timings(
            without_m0_times
        )

        m0_response_pair_rows = [
            {
                "route": "calc_m0_true",
                "plan_build_s": response_plan_with_m0_build_s,
                **with_m0_response_summary,
            },
            {
                "route": "calc_m0_false",
                "plan_build_s": response_plan_without_m0_build_s,
                **without_m0_response_summary,
            },
        ]

        m0_response_result = {
            "attempted": True,
            "supported": True,
            "error": None,
            "response_shape_with_m0": tuple(
                jnp.asarray(response_with_m0).shape
            ),
            "response_shape_without_m0": tuple(
                jnp.asarray(response_without_m0).shape
            ),
            "response_dtype_with_m0": str(
                jnp.asarray(response_with_m0).dtype
            ),
            "response_dtype_without_m0": str(
                jnp.asarray(response_without_m0).dtype
            ),
            "max_abs": response_comparison["max_abs"],
            "rel_l2": response_comparison["rel_l2"],
            "response_runtime_change_percent": 100.0 * (
                without_m0_response_summary["median_s"]
                / with_m0_response_summary["median_s"]
                - 1.0
            ),
            "response_speedup": (
                with_m0_response_summary["median_s"]
                / without_m0_response_summary["median_s"]
            ),
            "estimated_dwt_plus_response_with_m0_s": (
                summarize_timings(m0_true_pair_times)["median_s"]
                + with_m0_response_summary["median_s"]
            ),
            "estimated_dwt_plus_response_without_m0_s": (
                summarize_timings(m0_false_pair_times)["median_s"]
                + without_m0_response_summary["median_s"]
            ),
            "with_m0_shift_summary": with_m0_response_summary,
            "without_m0_shift_summary": without_m0_response_summary,
        }

        print("One-arm response timings:")
        show_rows(m0_response_pair_rows)

        print("\nOne-arm response comparison:")
        for key, value in m0_response_result.items():
            if key not in (
                "with_m0_shift_summary",
                "without_m0_shift_summary",
            ):
                print(f"  {key}: {value}")

    except Exception as exc:
        m0_response_result = {
            "attempted": True,
            "supported": False,
            "error": repr(exc),
        }
        print("The current response could not consume calc_m0=False:")
        print(m0_response_result["error"])
else:
    print("One-arm m=0 response check skipped.")



## 10. Host/device and dtype conversion costs

This separates the transform from:

- copying two time series to the JAX device;
- accepting device-resident input, when supported;
- converting the WDM outputs to the `complex64` representation used by the
  production time-shift kernel.

A device-input result is particularly relevant for future JAX-native waveform
generation and likelihood evaluation.


In [ ]:

# --- 12. Host/device transfer and output-cast diagnostics ---

conversion_rows = []


def copy_inputs_to_device():
    plus = jnp.asarray(hplus_td)
    cross = jnp.asarray(hcross_td)
    return plus, cross


(input_plus_device, input_cross_device), h2d_timing = benchmark_call(
    copy_inputs_to_device,
    warmups=1,
    repeats=REPEATS,
)
conversion_rows.append({"operation": "two_td_inputs_host_to_device", **h2d_timing})


def cast_outputs_to_c64():
    plus = jnp.asarray(w_plus_ref, dtype=jnp.complex64)
    cross = jnp.asarray(w_cross_ref, dtype=jnp.complex64)
    return plus, cross


(_, _), c64_timing = benchmark_call(
    cast_outputs_to_c64,
    warmups=1,
    repeats=REPEATS,
)
conversion_rows.append({"operation": "two_wdm_outputs_to_complex64", **c64_timing})

device_input_supported = False
device_input_error = None
device_input_timing = None

try:
    def run_two_device_inputs():
        return wdm.dwt(input_plus_device), wdm.dwt(input_cross_device)

    device_outputs, device_input_timing = benchmark_call(
        run_two_device_inputs,
        warmups=WARMUPS,
        repeats=REPEATS,
    )
    device_stack = jnp.stack(
        (jnp.asarray(device_outputs[0]), jnp.asarray(device_outputs[1])),
        axis=0,
    )
    device_comparison = compare_arrays(device_stack, reference_stack)
    device_input_supported = True
    conversion_rows.append({
        "operation": "two_dwt_device_inputs",
        **device_input_timing,
        **device_comparison,
    })
except Exception as exc:
    device_input_error = repr(exc)
    print("Device-resident DWT input unsupported or failed:")
    print(device_input_error)

conversion_table = show_rows(conversion_rows)

print("Device input supported:", device_input_supported)
if device_input_supported:
    print(
        "Device-input runtime change vs host-input baseline:",
        f"{100.0 * (device_input_timing['median_s'] / two_timing['median_s'] - 1.0):+.2f}%",
    )



## 11. Test batching and vectorisation without editing the package

Three independent candidates are tested:

1. passing a stacked `(2, N)` array directly to `wdm.dwt`;
2. `jax.jit(jax.vmap(wdm.dwt))`;
3. launching the two independent transforms concurrently from two Python
   threads.

Unsupported candidates are recorded rather than treated as notebook failures.
The threaded route is diagnostic only: each DWT may already use the full CPU,
so two concurrent transforms can easily compete for the same cores.


In [ ]:

# --- 15. Construct and validate experimental two-polarization routes ---

stacked_host = np.stack((hplus_td, hcross_td), axis=0)
candidate_functions = {}
candidate_validation_rows = []


def normalize_two_output(value):
    if isinstance(value, (tuple, list)) and len(value) == 2:
        return jnp.stack(
            (jnp.asarray(value[0]), jnp.asarray(value[1])),
            axis=0,
        )

    array = jnp.asarray(value)
    expected = tuple(reference_stack.shape)

    if tuple(array.shape) == expected:
        return array

    transposed_expected = (
        expected[1],
        expected[2],
        expected[0],
    )
    if tuple(array.shape) == transposed_expected:
        return jnp.transpose(array, (2, 0, 1))

    raise ValueError(
        f"Cannot normalize output shape {array.shape}; expected {expected} "
        f"or {transposed_expected}."
    )


def register_candidate(name, fn):
    try:
        output = fn()
        block_result(output)
        normalized = normalize_two_output(output)
        normalized.block_until_ready()
        comparison = compare_arrays(normalized, reference_stack)
        candidate_functions[name] = fn
        candidate_validation_rows.append({
            "candidate": name,
            "supported": True,
            "error": None,
            **comparison,
            "output_shape": tuple(normalized.shape),
            "output_dtype": str(normalized.dtype),
        })
    except Exception as exc:
        candidate_validation_rows.append({
            "candidate": name,
            "supported": False,
            "error": repr(exc),
            "max_abs": np.nan,
            "rel_l2": np.nan,
            "output_shape": None,
            "output_dtype": None,
        })


if RUN_NATIVE_BATCH_TEST:
    register_candidate(
        "native_stacked_input",
        lambda: wdm.dwt(stacked_host),
    )

if RUN_VMAP_TEST:
    try:
        vmapped_dwt = jax.jit(
            jax.vmap(lambda series: wdm.dwt(series), in_axes=0)
        )
        register_candidate(
            "jax_jit_vmap",
            lambda: vmapped_dwt(jnp.asarray(stacked_host)),
        )
    except Exception as exc:
        candidate_validation_rows.append({
            "candidate": "jax_jit_vmap",
            "supported": False,
            "error": repr(exc),
            "max_abs": np.nan,
            "rel_l2": np.nan,
            "output_shape": None,
            "output_dtype": None,
        })

if "THREAD_EXECUTOR" in globals():
    try:
        THREAD_EXECUTOR.shutdown(wait=True)
    except Exception:
        pass
    del THREAD_EXECUTOR

if RUN_THREAD_TEST:
    THREAD_EXECUTOR = ThreadPoolExecutor(max_workers=2)

    def run_two_threaded():
        plus_future = THREAD_EXECUTOR.submit(wdm.dwt, hplus_td)
        cross_future = THREAD_EXECUTOR.submit(wdm.dwt, hcross_td)
        return plus_future.result(), cross_future.result()

    register_candidate("two_python_threads", run_two_threaded)


candidate_validation_table = show_rows(candidate_validation_rows)
print("Supported candidates:", sorted(candidate_functions))



## 12. Paired timing against the sequential baseline

Each supported candidate receives two warm-ups and 15 paired timings. Execution
order alternates to reduce drift.

A candidate is regarded as a strong low-risk result only when:

- relative $L_2 \leq 10^{-6}$;
- median improvement is at least 3%;
- candidate $q_{75}$ is below baseline $q_{25}$.


In [ ]:

# --- 12. Paired baseline/candidate timings ---

paired_rows = []
paired_raw = {}

for candidate_name, candidate_fn in candidate_functions.items():
    print("\nCandidate:", candidate_name)

    for _ in range(WARMUPS):
        block_result(run_two_sequential_host())
        block_result(candidate_fn())

    baseline_times = []
    candidate_times = []

    for round_index in range(REPEATS):
        order = ("baseline", "candidate") if round_index % 2 == 0 else ("candidate", "baseline")

        for label in order:
            started = time.perf_counter()
            if label == "baseline":
                output = run_two_sequential_host()
                block_result(output)
                baseline_times.append(time.perf_counter() - started)
            else:
                output = candidate_fn()
                block_result(output)
                candidate_times.append(time.perf_counter() - started)

    candidate_output = normalize_two_output(candidate_fn())
    candidate_output.block_until_ready()
    comparison = compare_arrays(candidate_output, reference_stack)

    baseline_summary = summarize_timings(baseline_times)
    candidate_summary = summarize_timings(candidate_times)

    runtime_change_percent = 100.0 * (
        candidate_summary["median_s"] / baseline_summary["median_s"] - 1.0
    )
    improvement_percent = -runtime_change_percent
    iqr_separated = bool(
        candidate_summary["q75_s"] < baseline_summary["q25_s"]
    )
    numerically_valid = bool(
        np.isfinite(comparison["rel_l2"])
        and comparison["rel_l2"] <= MAX_REL_L2
    )

    decision = (
        "strong_candidate"
        if (
            numerically_valid
            and improvement_percent >= MIN_INTERESTING_IMPROVEMENT_PERCENT
            and iqr_separated
        )
        else "diagnostic_only"
    )

    row = {
        "candidate": candidate_name,
        "baseline_median_s": baseline_summary["median_s"],
        "candidate_median_s": candidate_summary["median_s"],
        "runtime_change_percent": runtime_change_percent,
        "speedup": baseline_summary["median_s"] / candidate_summary["median_s"],
        "baseline_q25_s": baseline_summary["q25_s"],
        "baseline_q75_s": baseline_summary["q75_s"],
        "candidate_q25_s": candidate_summary["q25_s"],
        "candidate_q75_s": candidate_summary["q75_s"],
        "iqr_separated": iqr_separated,
        "max_abs": comparison["max_abs"],
        "rel_l2": comparison["rel_l2"],
        "decision": decision,
    }
    paired_rows.append(row)
    paired_raw[candidate_name] = {
        "baseline_times_s": baseline_times,
        "candidate_times_s": candidate_times,
    }

paired_table = show_rows(paired_rows, sort_by="candidate_median_s")

if "THREAD_EXECUTOR" in globals():
    THREAD_EXECUTOR.shutdown(wait=True)
    del THREAD_EXECUTOR



## 13. Optional batch-throughput scaling

This section is disabled by default. Enable it only when the JIT-vmapped route
works and the two-polarization test suggests a meaningful benefit.

The goal is not only to batch plus/cross. In parameter estimation, the more
ambitious target is a batch of waveform proposals. Throughput per waveform is
therefore reported for batch sizes 1, 2, and 4.


In [ ]:

# --- 15. Optional vmap batch-throughput scaling ---

batch_scaling_rows = []

if RUN_BATCH_SCALING:
    if "jax_jit_vmap" not in candidate_functions:
        print("Skipping: jax_jit_vmap is not supported.")
    else:
        for batch_size in (1, 2, 4):
            base_series = np.stack((hplus_td, hcross_td), axis=0)
            repeats_needed = (batch_size + 1) // 2
            batch_host = np.tile(base_series, (repeats_needed, 1))[:batch_size]
            batch_device = jnp.asarray(batch_host)

            batch_fn = jax.jit(
                jax.vmap(lambda series: wdm.dwt(series), in_axes=0)
            )

            _, timing = benchmark_call(
                lambda batch_fn=batch_fn, batch_device=batch_device: batch_fn(batch_device),
                warmups=WARMUPS,
                repeats=max(5, REPEATS // 2),
            )

            batch_scaling_rows.append({
                "batch_size": batch_size,
                "median_batch_s": timing["median_s"],
                "median_per_waveform_s": timing["median_s"] / batch_size,
                "throughput_waveforms_per_s": batch_size / timing["median_s"],
                "iqr_batch_s": timing["iqr_s"],
                "input_mib": batch_host.nbytes / 1024**2,
            })

batch_scaling_table = show_rows(batch_scaling_rows)



## 14. Fresh-process physical-core scaling

The worker subprocess sets CPU affinity **before importing JAX** and measures
two sequential transforms. This is the closest available substitute for
hardware-counter profiling on a machine where `perf` cannot be installed.

Interpretation:

- strong scaling to the largest core count: more parallelism remains useful;
- early plateau with high effective CPU use: a shared resource such as memory,
  cache, or FFT synchronization is plausible;
- early plateau with low effective CPU use: the implementation may expose
  insufficient parallel work;
- slowdown at high core counts: thread overhead or shared-resource contention.

This remains an inference, not a direct memory-bandwidth measurement.


In [ ]:

# --- 14a. Detect physical-core topology and prepare the subprocess worker ---

def detect_physical_core_representatives():
    allowed = (
        sorted(os.sched_getaffinity(0))
        if hasattr(os, "sched_getaffinity")
        else list(range(os.cpu_count() or 1))
    )

    command = [
        "lscpu",
        "-p=CPU,CORE,SOCKET,NODE,ONLINE",
    ]
    completed = subprocess.run(command, capture_output=True, text=True)

    if completed.returncode != 0:
        print("lscpu topology query failed; treating allowed logical CPUs as distinct cores.")
        return allowed, []

    rows = []
    representatives = {}
    for line in completed.stdout.splitlines():
        if not line or line.startswith("#"):
            continue
        fields = line.split(",")
        if len(fields) < 5:
            continue
        cpu, core, socket, node, online = fields[:5]
        cpu = int(cpu)
        if cpu not in allowed:
            continue
        key = (int(socket), int(core))
        representatives.setdefault(key, cpu)
        rows.append({
            "cpu": cpu,
            "core": int(core),
            "socket": int(socket),
            "node": int(node) if node else -1,
            "online": online.strip().lower() in ("y", "yes", "1", "true"),
        })

    return sorted(representatives.values()), rows


PHYSICAL_CPU_IDS, CPU_TOPOLOGY_ROWS = detect_physical_core_representatives()
show_rows(CPU_TOPOLOGY_ROWS)

if REQUESTED_CORE_COUNTS is None:
    counts = []
    value = 1
    while value < len(PHYSICAL_CPU_IDS):
        counts.append(value)
        value *= 2
    counts.append(len(PHYSICAL_CPU_IDS))
    CPU_CORE_COUNTS = sorted(set(counts))
else:
    CPU_CORE_COUNTS = sorted(set(
        int(value)
        for value in REQUESTED_CORE_COUNTS
        if 1 <= int(value) <= len(PHYSICAL_CPU_IDS)
    ))

print("Physical-core representatives:", PHYSICAL_CPU_IDS)
print("Core counts:", CPU_CORE_COUNTS)

input_cache_path = CACHE_DIR / "full_year_wdm_transform_inputs.npz"
worker_path = CACHE_DIR / "wdm_transform_scaling_worker.py"

np.savez(
    input_cache_path,
    times_pad=np.asarray(times_pad),
    hplus_td=np.asarray(hplus_td),
    hcross_td=np.asarray(hcross_td),
)

worker_source = r"""
import argparse
import json
import os
import time
from pathlib import Path

parser = argparse.ArgumentParser()
parser.add_argument("--input", required=True)
parser.add_argument("--output", required=True)
parser.add_argument("--cpu-ids", required=True)
parser.add_argument("--warmups", type=int, default=2)
parser.add_argument("--repeats", type=int, default=9)
parser.add_argument("--Nf", type=int, required=True)
parser.add_argument("--q", type=int, required=True)
parser.add_argument("--d", type=int, required=True)
parser.add_argument("--A-frac", type=float, required=True)
args = parser.parse_args()

cpu_ids = [int(value) for value in args.cpu_ids.split(",") if value]
if hasattr(os, "sched_setaffinity"):
    os.sched_setaffinity(0, set(cpu_ids))

thread_count = str(max(1, len(cpu_ids)))
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["OMP_NUM_THREADS"] = thread_count
os.environ["MKL_NUM_THREADS"] = thread_count
os.environ["OPENBLAS_NUM_THREADS"] = thread_count
os.environ["NUMEXPR_NUM_THREADS"] = thread_count
os.environ["TF_NUM_INTRAOP_THREADS"] = thread_count
os.environ["TF_NUM_INTEROP_THREADS"] = "1"

import jax
import numpy as np

from WDM_LISAresponse.code.lisa_response.response_with_plan import (
    _build_wdm_dT_grid,
)


def block_result(value):
    for leaf in jax.tree_util.tree_leaves(value):
        if hasattr(leaf, "block_until_ready"):
            leaf.block_until_ready()


payload = np.load(args.input, mmap_mode="r")
times_pad = np.asarray(payload["times_pad"])
hplus_td = np.asarray(payload["hplus_td"])
hcross_td = np.asarray(payload["hcross_td"])

grid = _build_wdm_dT_grid(
    times_pad,
    Nf=args.Nf,
    d=args.d,
    A_frac=args.A_frac,
    q=args.q,
    calc_m0=True,
)
wdm = grid["wdm"]


def run_pair():
    plus = wdm.dwt(hplus_td)
    cross = wdm.dwt(hcross_td)
    block_result((plus, cross))
    return plus, cross


for _ in range(max(0, args.warmups)):
    run_pair()

wall_times = []
cpu_times = []
last = None

for _ in range(max(1, args.repeats)):
    wall_start = time.perf_counter()
    cpu_start = time.process_time()
    last = run_pair()
    cpu_times.append(time.process_time() - cpu_start)
    wall_times.append(time.perf_counter() - wall_start)

plus, cross = last
plus_np = np.asarray(plus)
cross_np = np.asarray(cross)

wall = np.asarray(wall_times, dtype=float)
cpu = np.asarray(cpu_times, dtype=float)
q25, q75 = np.percentile(wall, [25.0, 75.0])

result = {
    "cpu_ids": cpu_ids,
    "allowed_physical_cores": len(cpu_ids),
    "median_wall_s": float(np.median(wall)),
    "q25_wall_s": float(q25),
    "q75_wall_s": float(q75),
    "iqr_wall_s": float(q75 - q25),
    "median_cpu_s": float(np.median(cpu)),
    "median_effective_cores": float(np.median(cpu / wall)),
    "wall_times_s": wall_times,
    "cpu_times_s": cpu_times,
    "plus_shape": list(plus_np.shape),
    "cross_shape": list(cross_np.shape),
    "plus_dtype": str(plus_np.dtype),
    "cross_dtype": str(cross_np.dtype),
    "plus_norm": float(np.linalg.norm(plus_np)),
    "cross_norm": float(np.linalg.norm(cross_np)),
    "jax_backend": jax.default_backend(),
}

Path(args.output).write_text(json.dumps(result, indent=2), encoding="utf-8")
"""

worker_path.write_text(textwrap.dedent(worker_source), encoding="utf-8")

print("Input cache:", input_cache_path)
print("Worker:", worker_path)
print("Input cache size:", f"{input_cache_path.stat().st_size / 1024**2:.1f} MiB")


In [ ]:

# --- 14b. Run the physical-core scaling sweep ---

cpu_scaling_rows = []

if RUN_CPU_SCALING:
    for core_count in CPU_CORE_COUNTS:
        selected_cpu_ids = PHYSICAL_CPU_IDS[:core_count]
        output_path = CACHE_DIR / f"cpu_scaling_{core_count:03d}_cores.json"

        command = [
            sys.executable,
            str(worker_path),
            "--input",
            str(input_cache_path),
            "--output",
            str(output_path),
            "--cpu-ids",
            ",".join(str(cpu) for cpu in selected_cpu_ids),
            "--warmups",
            str(CPU_SCALING_WARMUPS),
            "--repeats",
            str(CPU_SCALING_REPEATS),
            "--Nf",
            str(Nf),
            "--q",
            str(q_wdm),
            "--d",
            str(d_wdm),
            "--A-frac",
            repr(float(A_frac)),
        ]

        print(f"Running {core_count} physical core(s):", selected_cpu_ids)
        completed = subprocess.run(
            command,
            capture_output=True,
            text=True,
        )

        if completed.returncode != 0:
            print("stdout:")
            print(completed.stdout)
            print("stderr:")
            print(completed.stderr)
            raise RuntimeError(
                f"CPU-scaling worker failed for {core_count} core(s)."
            )

        row = json.loads(output_path.read_text(encoding="utf-8"))
        cpu_scaling_rows.append(row)

    one_core_median = cpu_scaling_rows[0]["median_wall_s"]
    for row in cpu_scaling_rows:
        cores = row["allowed_physical_cores"]
        speedup = one_core_median / row["median_wall_s"]
        row["speedup_vs_1_core"] = speedup
        row["parallel_efficiency"] = speedup / cores
        row["effective_core_fraction"] = row["median_effective_cores"] / cores

cpu_scaling_table = show_rows([
    {
        "allowed_physical_cores": row["allowed_physical_cores"],
        "median_wall_s": row["median_wall_s"],
        "q25_wall_s": row["q25_wall_s"],
        "q75_wall_s": row["q75_wall_s"],
        "iqr_wall_s": row["iqr_wall_s"],
        "speedup_vs_1_core": row.get("speedup_vs_1_core", np.nan),
        "parallel_efficiency": row.get("parallel_efficiency", np.nan),
        "median_effective_cores": row["median_effective_cores"],
        "effective_core_fraction": row.get("effective_core_fraction", np.nan),
    }
    for row in cpu_scaling_rows
])

if cpu_scaling_rows:
    first = cpu_scaling_rows[0]
    last = cpu_scaling_rows[-1]
    print("\nFull-core speedup:", f"{last['speedup_vs_1_core']:.2f}x")
    print("Full-core efficiency:", f"{100.0 * last['parallel_efficiency']:.1f}%")
    print("Full-core effective fraction:", f"{100.0 * last['effective_core_fraction']:.1f}%")

    if len(cpu_scaling_rows) >= 2:
        previous = cpu_scaling_rows[-2]
        final_improvement = 100.0 * (
            1.0 - last["median_wall_s"] / previous["median_wall_s"]
        )
        print(
            "Improvement from penultimate to full core count:",
            f"{final_improvement:.1f}%",
        )

if plt is not None and cpu_scaling_rows:
    core_values = [row["allowed_physical_cores"] for row in cpu_scaling_rows]
    wall_values = [row["median_wall_s"] for row in cpu_scaling_rows]

    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    ax.plot(core_values, wall_values, marker="o")
    ax.set_xlabel("Allowed physical cores")
    ax.set_ylabel("Median two-DWT runtime [s]")
    ax.set_title("WDM transform physical-core scaling")
    ax.grid(True, alpha=0.3)
    plt.show()



## 15. Automatic synthesis and ambitious next steps

The notebook distinguishes three levels of possible change.

### Low-risk representation changes

- keep time-domain inputs on-device;
- keep plus/cross WDM outputs stacked on-device;
- avoid an unnecessary `complex128 -> complex64` host round-trip.

### Medium source-level changes

- add a true leading batch dimension to `dwt`;
- replace two separate FFT calls with one batched FFT;
- move Python loops or NumPy conversions into a JIT-traceable array program;
- reuse FFT/window workspaces across repeated evaluations.

### Ambitious architectural changes

- batch many waveform proposals, not only plus/cross;
- generate slowly evolving waveforms directly in the WDM domain;
- represent EMRI coefficients through a time-frequency track or mask rather
  than a dense full grid;
- fuse direct WDM-domain waveform generation with the response input
  representation.

Dense memory pressure has **not** already been ruled out. The time-shift scaling
result applied to a different kernel, and the failed contiguous-$T_l/T_p$
experiment only showed that those two small arrays were not the bottleneck.


In [ ]:

# --- 15. Summarise the evidence and save the compact results ---

print("=== WDM transform architecture summary ===\n")

print("Baseline two-transform median:", f"{two_timing['median_s']:.4f} s")
print("Baseline two-transform IQR:", f"{two_timing['iqr_s']:.4f} s")
print("Input dtype:", hplus_td.dtype)
print("Output dtype:", getattr(w_plus_ref, "dtype", None))
print("Output location:", array_record("w_plus_ref", w_plus_ref)["location"])
print("Persistent array-like WDM state:", f"{persistent_wdm_mib:.1f} MiB")

print("\ncalc_m0 / q study:")

if m0_study_error is not None:
    print("  transform comparison failed:", m0_study_error)
elif m0_comparison:
    print(
        "  two-DWT runtime change:",
        f"{m0_comparison['runtime_change_percent']:+.2f}%",
    )
    print(
        "  plus m=0 power fraction:",
        f"{m0_comparison['with_m0_plus_power_fraction']:.6e}",
    )
    print(
        "  cross m=0 power fraction:",
        f"{m0_comparison['with_m0_cross_power_fraction']:.6e}",
    )
    print(
        "  plus nonzero alignment:",
        m0_comparison["plus_nonzero_alignment"]["best_convention"],
        f"rel_l2={m0_comparison['plus_nonzero_alignment']['best_rel_l2']:.3e}",
    )
    print(
        "  cross nonzero alignment:",
        m0_comparison["cross_nonzero_alignment"]["best_convention"],
        f"rel_l2={m0_comparison['cross_nonzero_alignment']['best_rel_l2']:.3e}",
    )

if m0_response_result.get("attempted", False):
    if m0_response_result.get("supported", False):
        print(
            "  one-arm rel_l2 without m=0:",
            f"{m0_response_result['rel_l2']:.3e}",
        )
        print(
            "  estimated DWT+response runtime change:",
            f"{100.0 * (m0_response_result['estimated_dwt_plus_response_without_m0_s'] / m0_response_result['estimated_dwt_plus_response_with_m0_s'] - 1.0):+.2f}%",
        )
    else:
        print(
            "  current one-arm response rejected the no-m0 representation:",
            m0_response_result.get("error"),
        )

if paired_rows:
    print("\nCandidate routes:")
    for row in sorted(paired_rows, key=lambda item: item["candidate_median_s"]):
        print(
            f"  {row['candidate']}: "
            f"{row['candidate_median_s']:.4f} s, "
            f"{row['runtime_change_percent']:+.2f}%, "
            f"rel_l2={row['rel_l2']:.3e}, "
            f"{row['decision']}"
        )
else:
    print("\nNo alternative public-API batching route was successfully timed.")

if cpu_scaling_rows:
    full = cpu_scaling_rows[-1]
    print("\nCPU scaling:")
    print(
        f"  1 core -> {cpu_scaling_rows[0]['median_wall_s']:.4f} s; "
        f"{full['allowed_physical_cores']} cores -> {full['median_wall_s']:.4f} s"
    )
    print(
        f"  speedup={full['speedup_vs_1_core']:.2f}x; "
        f"efficiency={100.0 * full['parallel_efficiency']:.1f}%; "
        f"effective fraction={100.0 * full['effective_core_fraction']:.1f}%"
    )

print("\nDecision guide:")

strong_candidates = [
    row for row in paired_rows
    if row["decision"] == "strong_candidate"
]

if strong_candidates:
    best = min(strong_candidates, key=lambda row: row["candidate_median_s"])
    print(
        f"- The current public API already exposes a strong candidate: "
        f"{best['candidate']}. Confirm it in an end-to-end response benchmark."
    )
else:
    print(
        "- No current public-API vectorisation route has yet passed the "
        "3% and IQR-separation guard."
    )

if m0_comparison:
    plus_nonzero = m0_comparison["plus_nonzero_alignment"]["best_rel_l2"]
    cross_nonzero = m0_comparison["cross_nonzero_alignment"]["best_rel_l2"]
    if (
        np.isfinite(plus_nonzero)
        and np.isfinite(cross_nonzero)
        and plus_nonzero <= MAX_REL_L2
        and cross_nonzero <= MAX_REL_L2
    ):
        print(
            "- calc_m0=False preserves the aligned nonzero-frequency "
            "coefficients to the configured tolerance. Response-level "
            "validation now determines whether the m=0 layer can be removed."
        )
    else:
        print(
            "- calc_m0 changes more than a removable first column. Do not "
            "change the production convention without understanding the "
            "basis/indexing difference."
        )

if source_signals.get("python_for", False) or source_signals.get("numpy_conversion", False):
    print(
        "- Source inspection found Python-loop and/or NumPy-conversion signals. "
        "A JIT-traceable batched DWT refactor is a plausible medium-sized project."
    )
else:
    print(
        "- Source inspection did not expose an obvious Python-loop/NumPy boundary "
        "in the public dwt method; inspect the private transform methods printed above."
    )

if cpu_scaling_rows:
    full = cpu_scaling_rows[-1]
    if full["parallel_efficiency"] < 0.55 and full["effective_core_fraction"] > 0.55:
        print(
            "- Scaling loses efficiency while maintaining substantial CPU use. "
            "A shared FFT/cache/memory resource is plausible, but not proven."
        )
    elif full["effective_core_fraction"] < 0.45:
        print(
            "- Effective CPU use is low at full affinity. More exposed batch "
            "parallelism or a source-level vectorisation may be valuable."
        )
    else:
        print(
            "- The transform continues to use a substantial fraction of the CPU. "
            "Prioritise batched FFT/workspace reuse over simply adding more threads."
        )

print(
    "- The ambitious scientific route remains direct WDM-domain waveform "
    "generation and track/mask representations. That changes the amount of "
    "data produced, rather than only accelerating the dense transform."
)

results = {
    "environment": {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "jax": jax.__version__,
        "jax_backend": jax.default_backend(),
        "wdm_file": WDM.__file__,
        "wdm_lisaresponse_file": WDM_LISAresponse.__file__,
    },
    "configuration": {
        "N": N,
        "dt": dt,
        "Nf": Nf,
        "q_wdm": q_wdm,
        "d_wdm": d_wdm,
        "A_frac": A_frac,
        "m0_q_values": M0_Q_VALUES,
    },
    "representation_rows": representation_rows,
    "m0_study_error": m0_study_error,
    "m0_timing_rows": m0_timing_rows,
    "m0_alignment_rows": m0_alignment_rows,
    "m0_comparison": m0_comparison,
    "m0_q_rows": m0_q_rows,
    "m0_response_pair_rows": m0_response_pair_rows,
    "m0_response_result": m0_response_result,
    "baseline_rows": baseline_rows,
    "conversion_rows": conversion_rows,
    "candidate_validation_rows": candidate_validation_rows,
    "paired_rows": paired_rows,
    "paired_raw": paired_raw,
    "batch_scaling_rows": batch_scaling_rows,
    "cpu_scaling_rows": cpu_scaling_rows,
}

results_path = CACHE_DIR / "wdm_transform_architecture_results.json"
results_path.write_text(json.dumps(results, indent=2, default=str), encoding="utf-8")
print("\nSaved:", results_path)



## 16. Outputs to bring back for interpretation

The most useful outputs are:

1. the `dwt` source and structural-signal block;
2. the representation inventory;
3. the baseline plus/cross/two-transform timing table;
4. the `calc_m0=True` versus `calc_m0=False` timing and alignment tables;
5. the constructor-`q` sweep;
6. the optional one-arm response comparison;
7. the host/device conversion table;
8. the candidate validation and paired-timing tables;
9. the physical-core scaling table;
10. `.benchmark_cache/wdm_transform_profile/wdm_transform_architecture_results.json`.

Do not change the production `calc_m0` convention based only on transform
runtime. A no-\(m=0\) candidate should preserve the intended \(m\geq1\)
coefficients, work with the prepared response, and then pass the normal
time-domain/TDI mismatch validation.
